#  Classification and logistic regression

In this exercise, I predicted a binary outcome using a linear probability model and a regularized logit.

Specifically, I made a personal movie recommendation engine by combining data on individual films from IMDB with responses from my classmate on their personal film ratings

## Setup

Import the following libraries:

In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
import sklearn
import matplotlib.pyplot as plt
import seaborn as sns

## Data import

First, I imported the individual Google sheets each classmate filled out and combine them into a single dataframe.

Unfortunately, this data was not copied over to my new computer. But I still have the code below

In [ ]:
# ============================
# Generate a list of the CSV files containing each person's individual movie ratings
filepath=r'C:\Users\mattv\OneDrive\Documents\Academics\University of Southern California\04 - Fall 2022\03 - PPD 599 Special Topics (Python)\In-Class-Exercises\Data'
import os

# Use OS command "listdir" to get a list of all files in the filepath
all_files = os.listdir(filepath)

# use a list comprehension to create a list of those files
# that are CSV only
csv_list = [ x for x in all_files if x.endswith('.csv') ]

Import each file and combine them into a single large file.

In [ ]:
# Import the first file in the list to a dataframe
filename=filepath+csv_list[0]
df=pd.read_csv(filename)

# Append the remaining files to the dataframe
# using .concat()
for file in csv_list[1:len(csv_list)]:
    filename=filepath+file
    df2=pd.read_csv(filename)
    df=pd.concat([df,df2])
    

In [ ]:
df

In [ ]:
# Check that usernames are balanced
df.loc[:,'Username'].value_counts()

## Prepare the movie-rating data

Create a set of dummies equal to 1 if the rater liked the movie, and another equal to 1 if they disliked it. (The omitted case is that the movie was not rated.)

In [ ]:
df['Liked']=(df['Rating']=="Liked it")*1
df['Disliked']=(df['Rating']=="Disliked it")*1

Keep the identifying information and these ratings.

In [ ]:
df=df[['IMDb ID','Username','Film','Liked','Disliked']]    

Transform the ratings data in two important further ways:
1. Go from each row equaling one rating *of* one movie to being *one movie*, with everybody's ratings as a separate column
2. Define a target variable for your own ratings.

First, use the identifying variables IMDb ID, Film title, and rater's username to define a MultiIndex

In [ ]:
df=df.set_index(['IMDb ID','Film','Username'])
df

Use `.unstack` to turn the rater's name into a column concept (and to reshape the data.). 

In [ ]:
df=df.unstack(2)

df

Turn the column multiindex into a regular set of column names, and turn the row index back into a single, ID-based index, as follows:

In [ ]:
df.columns = ['_'.join(col) for col in df.columns.values]

df=df.reset_index().set_index('IMDb ID')

This should have "flattened" the column multiindex into columns.

In [ ]:
df

Create a new variable, "target" equal to 1 if I liked the film, 0 if I disliked it, and NaN (missing) if you didn't rate it. Then, drop the original, unconverted data on your likes and dislikes. 

In [ ]:
df['target']=np.nan
df.loc[df['Liked_Matt']==1,'target']=1
df.loc[df['Disliked_Matt']==1,'target']=0

df=df.drop(['Liked_Matt','Disliked_Matt'], axis=1)

In [ ]:
df['target'].value_counts()

## Merge in other film information

Ddd the other film information scraped earlier in the semester to get a rich set of features

In [ ]:
# Change the file path as needed
imdb_file=r'C:\Users\mattv\OneDrive\Documents\Academics\University of Southern California\04 - Fall 2022\03 - PPD 599 Special Topics (Python)\In-Class-Exercises\Data\imdb_data.csv'

In [ ]:
imdb_data=(pd.read_csv(imdb_file)
            .rename(columns={'id':'IMDb ID'})
            .set_index('IMDb ID')
        )

Define a command that will turn IMDb strings about monetary quantities, like "$50,000 (estimated)", into numbers, like "50000"

In [ ]:
def money_to_numeric(seriesname):
    
    new_series=seriesname.str.replace(',','',regex=False)
    new_series=new_series.str.replace('$','',regex=False)
    new_series=new_series.str.replace(' (estimated)','',regex=False)
    new_series=pd.to_numeric(new_series, errors='coerce')
    
    return new_series

Convert the four financial variables to numbers

In [ ]:
imdb_data['opening']=money_to_numeric(imdb_data['openingWeekendUSA'])
imdb_data['budget']=money_to_numeric(imdb_data['budget'])
imdb_data['worldwide']=money_to_numeric(imdb_data['cumulativeWorldwideGross'])
imdb_data['us_gross']=money_to_numeric(imdb_data['grossUSA'])

Create a new variable that turns the column with a list of film genres, stored as a string, into a Python list.

E.g. "Action,Adventure,Comedy" --> ['Action', 'Adventure', 'Comedy']

In [ ]:
imdb_data['genre_list']=imdb_data['genres'].str.replace(' ','',regex=False).str.split(',')

imdb_data['genre_list'].value_counts()

Convert the genre list to a set of dummy variables for each genre.

In [ ]:
genre_data=imdb_data.genre_list.str.join('|').str.get_dummies().add_prefix('gnr_')   
    
imdb_data=imdb_data.join(genre_data)    

In [ ]:
genre_list=genre_data.columns

Inspect the IMDb data.

In [ ]:
imdb_data

## Create the final merged dataset

Keep completely observed data for identifying the film, genre information, financial performance, and IMDb community scores.

In [ ]:
keeplist=(['year','imDbRating','imDbRatingVotes','opening','worldwide','us_gross','budget',
            'gnr_Action', 'gnr_Adventure', 'gnr_Animation', 'gnr_Biography',
            'gnr_Comedy', 'gnr_Crime', 'gnr_Documentary', 'gnr_Drama', 'gnr_Family',
            'gnr_Fantasy', 'gnr_History', 'gnr_Horror', 'gnr_Music', 'gnr_Musical',
            'gnr_Mystery', 'gnr_Romance', 'gnr_Sci-Fi', 'gnr_Sport',
            'gnr_Thriller', 'gnr_Western']) #'gnr_News', 'gnr_War', 
imdb_data=imdb_data[keeplist].dropna()

Join the IMDb data to our main dataframe using the IMDb ID as the row index

In [ ]:
df=df.join(imdb_data, how='inner')

In [ ]:
df

Create a list of the variables that are features (regressors), not the outcome and not identifying. This list comprehension will keep every column in our merged data frame except for 'target' and 'Film'.

In [ ]:
list_of_features=[i for i in df.columns.to_list() if i not in ['target','Film']]

Lastly, split the data into subsets where I rated the movie (which we will use to create a model) and ones where I didn't (which we will use to suggest films I should watch).

In [ ]:
df_observed=df.loc[df['target'].notnull()].reset_index()

df_notobserved=df.loc[df['target'].isnull()].reset_index()

In [ ]:
df_observed.head(5)

In [ ]:
df_notobserved.head(5)

## Linear Probability Model

Let's predict which movies you would like with a linear probability model. Use statsmodels to add a constant to the list of features for the dataframe of *observed* target values, and predict which movies you do and do not like.

In [ ]:
# Create a matrix of the variables of interest and add a constant
X=sm.add_constant(df_observed[list_of_features])

# Create the target vector
y=df_observed['target']

# Fit the model
lpm_reg=sm.OLS(y,X).fit()

Look at the results.

In [ ]:
print(lpm_reg.summary())

## Standard Logit doesn't run on my limited data

In [ ]:
logit_reg=sm.Logit(y,X).fit()
print(logit_reg.summary())

## Machine-learning Logit

Import logistic regression from scikit-learn and set it to use a Lasso (L1) penalty with the liblinear solver. Use this model to predict which movies I would like.

In [ ]:
# Define and fit a machine learning logit
from sklearn.linear_model import LogisticRegression
logit_model=LogisticRegression(penalty='l1', 
                                 random_state=0, 
                                 solver='liblinear',
                                 max_iter=1_000)

Fit the model.

In [ ]:
# Fit the logit using the same features and outcome you defined for the LPM
movie_lasso=logit_model.fit(X,y)

## Predict movies I should watch

Add a constant to the features from the subset of the data where you I didn't record an opinion. Predict the probability that I will like a film based on linear probability model. 

In [ ]:
# Create features+constant from non-observed dataframe, and predict movie recommendations
df_notobserved['lpm_predictions']=lpm_reg.predict(sm.add_constant(df_notobserved[list_of_features]))

# Sort the dataframe from the movies you're most likely to like to the least likely
df_notobserved=df_notobserved.sort_values('lpm_predictions', ascending=False)

# Look at the first ten observatios of the sorted dataframe.
df_notobserved[['Film','lpm_predictions']].head(10)



Repeat this for your lasso model

In [2]:
# Extract predicted probabilities for the Lasso model and the films I haven't seen.
# Remember that this is the second column of the .predict_proba() function
df_notobserved['lasso_prediction']=movie_lasso.predict_proba(
                        sm.add_constant(df_notobserved[list_of_features]))[:,1]

# Sort
df_notobserved=df_notobserved.sort_values('lasso_prediction', ascending=False)

# Look at the ten most-recommended
df_notobserved[['Film','lasso_prediction']].head(10)


NameError: name 'movie_lasso' is not defined

Use seaborn to make a scatterplot of the linear probability predictions vs. the logit predictions.

In [ ]:
sns.scatterplot(data=df_notobserved,x='lpm_predictions',y='lasso_prediction')

plt.show()